# 1. Setup and Data Loading

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import torch
import torchvision

#Little commonly used shortcut
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision import transforms
# --- Importations modifiées ---
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torchvision.models import swin_t, Swin_T_Weights
import torch
import torch.nn as nn
import torch.optim as optim
import torch
import torch.nn.functional as F
import torch
import torch.nn as nn

from torchvision.models import efficientnet_v2_m, EfficientNet_V2_M_Weights
import torch
import torch.nn as nn
from torchvision.models import swin_t, Swin_T_Weights # Swin_T est Swin Transformer Tiny
import torch
import torch.nn as nn
from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import pandas as pd

In [ ]:
# Define the root directory for the dataset (make sure to adjust this path locally)
DATA_ROOT = "./data" 
train_dir = os.path.join(DATA_ROOT, "training_images")

# Define basic data augmentations for the training set
transform =  transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.RandomPerspective(0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load the dataset
dataset = datasets.ImageFolder(train_dir, transform=transform)

# Split train / validation
train_size = int(0.85 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])
# Dataloaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False)



print("Classes :", dataset.classes)
print("Nb images :", len(dataset))

Classes : ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']
Nombre total d’images : 4279


In [4]:


# Transformations (les mêmes que pour le training set)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],
                         std=[0.229,0.224,0.225])
])
class TestDataset(Dataset):
    def __init__(self, folder, transform=None):
        self.transform = transform
        # Liste de toutes les images dans le dossier
        self.images = [os.path.join(folder, f) for f in os.listdir(folder) 
                       if f.endswith('.jpg') or f.endswith('.png')]
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image  # Pas de label ici
# Création du dataset et dataloader
test_dataset = TestDataset("/kaggle/input/polytech-nice-deep-learning-course-2025/RealWaste/test_images", transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Nombre d'images test :", len(test_dataset))

Nombre d'images test : 473


# 2. Model Initialization

In [ ]:

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


weights = EfficientNet_V2_M_Weights.DEFAULT  
model_Efficient = efficientnet_v2_m(weights=weights) 


num_features = model_Efficient.classifier[1].in_features
model_Efficient.classifier[1] = nn.Linear(num_features, 9)


model_Efficient = model_Efficient.to(device)



Using device: cuda


Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:00<00:00, 221MB/s] 



Modèle chargé et modifié avec succès : EfficientNetV2-M
La dernière couche a maintenant 9 sorties (9 classes).


# 3. Training and Fine-Tuning Strategy

In [21]:
# --- Étape de Dégel COMPLET du Modèle ---

# 1. Parcourir TOUS les paramètres du modèle
print("Décongélation de TOUS les paramètres du modèle pour le Full Fine-Tuning...")
for param in model_Efficient.parameters():
    param.requires_grad = True

# 2. Vérification
total_trainable_params = sum(p.numel() for p in model_Efficient.parameters() if p.requires_grad)
print(f"✅ Tous les paramètres sont maintenant dégelés.")
print(f"Nombre total de paramètres entraînables : {total_trainable_params}")

Décongélation de TOUS les paramètres du modèle pour le Full Fine-Tuning...
✅ Tous les paramètres sont maintenant dégelés.
Nombre total de paramètres entraînables : 52869885


In [22]:
# --- Le reste est identique ---
optimizer_Efficient = optim.AdamW(model_Efficient.parameters(), lr=2e-5, weight_decay=1e-4)
scheduler_Efficient = torch.optim.lr_scheduler.StepLR(optimizer_Efficient, step_size=5, gamma=0.5)
criterion_Efficient = nn.CrossEntropyLoss()

In [20]:
model_ef2=model_Efficient 

In [ ]:


best_val_acc_Efficient = 0.0  # 🔹 initialisation
N_EPOCHS = 20

for epoch in range(N_EPOCHS):
    model_Efficient.train()
    running_loss = 0.0
    running_corrects = 0
    total_samples = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer_Efficient.zero_grad()
        outputs = model_Efficient(images)
        loss = criterion_Efficient(outputs, labels)
        loss.backward()
        optimizer_Efficient.step()

        running_loss += loss.item() * images.size(0)
        running_corrects += (outputs.argmax(1) == labels).sum().item()
        total_samples += images.size(0)

    epoch_loss = running_loss / total_samples
    epoch_acc = running_corrects / total_samples

    # --- Validation ---
    model_Efficient.eval()
    val_loss, val_corrects, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_Efficient(images)
            loss = criterion_Efficient(outputs, labels)
            val_loss += loss.item() * images.size(0)
            val_corrects += (outputs.argmax(1) == labels).sum().item()
            val_total += images.size(0)
    val_loss /= val_total
    val_acc = val_corrects / val_total

    print(f"Epoch {epoch+1}/{N_EPOCHS} - Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")

    # --- Sauvegarde du meilleur modèle ---
    if val_acc > best_val_acc_Efficient:
        best_val_acc_Efficient = val_acc
        torch.save(model_Efficient.state_dict(), "best_model_Efficient.pth")
        print(f" Nouveau meilleur modèle sauvegardé (Val Acc: {best_val_acc_Efficient:.4f})")

    # --- Scheduler ---
    print(f"Learning Rate actuel : {scheduler_Efficient.get_last_lr()[0]:.6f}")
    scheduler_Efficient.step()

# --- Évaluation finale ---
from sklearn.metrics import confusion_matrix, classification_report

model_Efficient.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_Efficient(images)
        preds = outputs.argmax(1)
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

print("=== MATRICE DE CONFUSION ===")
print(confusion_matrix(y_true, y_pred))

print("\n=== RAPPORT DE CLASSIFICATION ===")
print(classification_report(y_true, y_pred, target_names=dataset.classes))

weights_path = "best_model_Efficient.pth"

model_Efficient.load_state_dict(torch.load(weights_path))

Epoch 1/20 - Train Loss: 0.2299, Train Acc: 0.9222, Val Loss: 0.3837, Val Acc: 0.8660
 Nouveau meilleur modèle sauvegardé (Val Acc: 0.8660)
Learning Rate actuel : 0.000020
Epoch 2/20 - Train Loss: 0.2133, Train Acc: 0.9260, Val Loss: 0.2971, Val Acc: 0.8972
 Nouveau meilleur modèle sauvegardé (Val Acc: 0.8972)
Learning Rate actuel : 0.000020
Epoch 3/20 - Train Loss: 0.1734, Train Acc: 0.9428, Val Loss: 0.2672, Val Acc: 0.9174
 Nouveau meilleur modèle sauvegardé (Val Acc: 0.9174)
Learning Rate actuel : 0.000020
Epoch 4/20 - Train Loss: 0.1652, Train Acc: 0.9445, Val Loss: 0.2684, Val Acc: 0.9159
Learning Rate actuel : 0.000020
Epoch 5/20 - Train Loss: 0.1650, Train Acc: 0.9412, Val Loss: 0.2718, Val Acc: 0.9065
Learning Rate actuel : 0.000020
Epoch 6/20 - Train Loss: 0.1521, Train Acc: 0.9491, Val Loss: 0.3031, Val Acc: 0.9050
Learning Rate actuel : 0.000010
Epoch 7/20 - Train Loss: 0.1266, Train Acc: 0.9557, Val Loss: 0.2731, Val Acc: 0.9190
 Nouveau meilleur modèle sauvegardé (Val Acc

In [ ]:
model_Efficient.eval()
predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)   # ← Important !
        outputs = model_Efficient(images)
        preds = outputs.argmax(1)
        predictions.extend(preds.cpu().tolist())  # on récupère sur CPU pour la liste

print("Prédictions pour le test set :", predictions)

model_Efficient.eval()
val_loss, val_corrects, val_total = 0.0, 0, 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model_Efficient(images)
        loss = criterion_Efficient(outputs, labels)
        val_loss += loss.item() * images.size(0)
        val_corrects += (outputs.argmax(1) == labels).sum().item()
        val_total += images.size(0)

val_loss /= val_total
val_acc = val_corrects / val_total
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc:.4f}")


# On récupère les noms de fichiers dans le test dataset
filenames = [os.path.basename(path) for path in test_dataset.images]

# On convertit les indices prédits en noms de classes
predicted_labels = [dataset.classes[idx] for idx in predictions]

# Création du DataFrame pour Kaggle
submission_df = pd.DataFrame({
    "filename": filenames,
    "category": predicted_labels
})

# Sauvegarde en CSV
submission_df.to_csv("submission.csv", index=False)
print("CSV de soumission créé : submission.csv")


Prédictions pour le test set : [3, 0, 1, 5, 6, 4, 0, 6, 3, 7, 2, 3, 6, 8, 6, 4, 6, 0, 1, 3, 3, 6, 3, 0, 2, 2, 0, 0, 3, 0, 4, 2, 6, 8, 3, 4, 4, 6, 2, 6, 3, 0, 4, 3, 4, 4, 7, 3, 5, 4, 3, 0, 1, 8, 1, 6, 2, 5, 6, 6, 0, 6, 1, 8, 2, 2, 7, 2, 3, 3, 1, 2, 0, 3, 5, 5, 1, 6, 5, 6, 5, 6, 5, 3, 6, 0, 4, 6, 0, 8, 3, 1, 4, 1, 5, 4, 7, 5, 2, 4, 1, 4, 7, 0, 6, 3, 0, 8, 3, 0, 5, 5, 7, 5, 8, 4, 2, 5, 8, 7, 6, 8, 7, 3, 3, 6, 0, 6, 0, 6, 5, 4, 8, 1, 2, 8, 2, 6, 2, 1, 3, 3, 0, 5, 4, 6, 0, 5, 5, 1, 4, 2, 7, 3, 4, 0, 2, 8, 7, 7, 1, 0, 6, 4, 7, 4, 2, 8, 3, 2, 3, 2, 0, 3, 6, 7, 2, 5, 3, 6, 7, 1, 5, 4, 3, 6, 1, 0, 8, 6, 7, 1, 7, 8, 8, 0, 6, 6, 6, 6, 8, 2, 6, 3, 1, 2, 6, 4, 5, 4, 2, 8, 3, 1, 8, 1, 7, 6, 6, 5, 3, 3, 6, 3, 2, 7, 2, 2, 3, 3, 6, 8, 8, 2, 3, 6, 3, 4, 7, 1, 8, 3, 6, 6, 6, 6, 3, 3, 6, 0, 3, 6, 1, 5, 6, 8, 5, 3, 6, 6, 3, 3, 2, 0, 3, 8, 2, 0, 8, 3, 2, 5, 6, 3, 6, 8, 1, 1, 6, 8, 0, 1, 6, 1, 3, 6, 1, 8, 1, 2, 2, 5, 4, 1, 6, 8, 8, 1, 0, 3, 0, 5, 0, 8, 6, 5, 6, 4, 8, 6, 7, 6, 6, 0, 8, 6, 5, 5, 0, 5, 1, 6, 2,

# 4. Weighted Ensemble and Final Evaluation

In [ ]:

# --- 1. Configuration et Instanciation du Modèle (model_swin) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instancier le modèle Swin Transformer Tiny
weights = Swin_T_Weights.DEFAULT
model_swin = swin_t(weights=weights)

# Adapter la dernière couche à 9 classes : Swin utilise 'head'
num_features = model_swin.head.in_features
model_swin.head = nn.Linear(num_features, 9)

# Mettre le modèle sur le bon périphérique
model_swin = model_swin.to(device)


#Chargement du model déjà entrainé 
weights_path = "/kaggle/input/best-model-swin-pth/pytorch/default/1/best_model_SwinT.pth"
model_swin.load_state_dict(torch.load(weights_path, map_location=device))

In [ ]:

#Configuration et Instanciation du Modèle (model_cn)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instancier le modèle ConvNeXt Tiny
weights = ConvNeXt_Tiny_Weights.DEFAULT
model_cn = convnext_tiny(weights=weights)

# Adapter la dernière couche à 9 classes : ConvNeXt utilise 'classifier[2]'
num_features = model_cn.classifier[2].in_features
model_cn.classifier[2] = nn.Linear(num_features, 9)

# Mettre le modèle sur le bon périphérique
model_cn = model_cn.to(device)


# Chargement du model déjà entrainé
weights_path = "/kaggle/input/best-model-cn-pth/pytorch/default/1/best_model_ConvNeXt.pth"
model_cn.load_state_dict(torch.load(weights_path, map_location=device))


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

criterion = nn.CrossEntropyLoss()
criterion_Efficient = criterion 
criterion_SwinT = criterion     
print("Loss functions (CrossEntropyLoss) defined.")


# --- FONCTION D'ÉVALUATION ---
def calculate_val_metrics(model, data_loader, device, criterion_func):
    """Calcule la perte et la précision d'un modèle sur un jeu de données."""
    model.eval()
    running_loss, running_corrects, total = 0.0, 0, 0
    
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion_func(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            running_corrects += (outputs.argmax(1) == labels).sum().item()
            total += images.size(0)
            
    loss = running_loss / total
    acc = running_corrects / total
    return loss, acc

Using device: cuda
Loss functions (CrossEntropyLoss) defined.


In [ ]:
model_cn.eval()
model_Efficient.eval()
model_swin.eval()
print("\nLes modèles sont prêts pour l'évaluation et la pondération.")


# --- ÉVALUATION ET CALCUL DES POIDS DYNAMIQUES ---

print(" Évaluation des modèles sur Validation pour le calcul des poids...")

loss_conv, acc_conv = calculate_val_metrics(model_cn, val_loader, device, criterion)
loss_eff, acc_eff = calculate_val_metrics(model_Efficient, val_loader, device, criterion_Efficient)
loss_swin, acc_swin = calculate_val_metrics(model_swin, val_loader, device, criterion_SwinT)

# Affichage des scores de base
print("\n===  Scores Validation Individuels ===")
print(f"ConvNeXt        → Val Loss: {loss_conv:.4f}, Val Acc: {acc_conv:.4f}")
print(f"EfficientNet-B4 → Val Loss: {loss_eff:.4f}, Val Acc: {acc_eff:.4f}")
print(f"Swin-T          → Val Loss: {loss_swin:.4f}, Val Acc: {acc_swin:.4f}")


w_conv = acc_conv
w_eff  = acc_eff
w_swin = acc_swin

# Normalisation des poids
s = w_conv + w_eff + w_swin
w_conv_norm, w_eff_norm, w_swin_norm = w_conv/s, w_eff/s, w_swin/s

print(f"\n→ Poids Normalisés: ConvNeXt={w_conv_norm:.4f}, EfficientNet={w_eff_norm:.4f}, Swin-T={w_swin_norm:.4f}")


# --- VALIDATION DE L'ENSEMBLE PONDÉRÉ ---
ensemble_correct, total = 0, 0

print("\n Validation de l'ensemble pondéré...")
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)

        # Sorties brutes
        out_conv = model_cn(images)
        out_swin = model_swin(images)
        out_eff  = model_Efficient(images)

        # Convertir en probabilités (softmax)
        probs_conv = F.softmax(out_conv, dim=1)
        probs_swin = F.softmax(out_swin, dim=1)
        probs_eff  = F.softmax(out_eff, dim=1)

        # Moyenne pondérée (utilisation des poids normalisés)
        ensemble_probs = (
            w_conv_norm * probs_conv +
            w_swin_norm * probs_swin +
            w_eff_norm  * probs_eff
        )

        preds = ensemble_probs.argmax(dim=1)
        ensemble_correct += (preds == labels).sum().item()
        total += labels.size(0)

ensemble_acc = ensemble_correct / total
print(f" Accuracy de l'ensemble pondéré (Val): {ensemble_acc:.4f}")


# --- PRÉDICTION FINALE SUR L'ENSEMBLE DE TEST ---
predictions = []

print("\n Génération des prédictions sur l'ensemble de test...")
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)

        # Sorties brutes
        out_conv = model_cn(images)
        out_eff  = model_Efficient(images)
        out_swin = model_swin(images)

        # Probabilités softmax
        probs_conv = F.softmax(out_conv, dim=1)
        probs_eff  = F.softmax(out_eff, dim=1)
        probs_swin = F.softmax(out_swin, dim=1)

        # Moyenne pondérée (on réutilise les mêmes poids normalisés)
        ensemble_probs = (
            w_conv_norm * probs_conv + 
            w_eff_norm  * probs_eff + 
            w_swin_norm * probs_swin
        )

        # Prédiction finale
        preds = ensemble_probs.argmax(dim=1)
        predictions.extend(preds.cpu().tolist())

# Création du CSV
filenames = [os.path.basename(path) for path in test_dataset.images]
predicted_labels = [dataset.classes[idx] for idx in predictions]

submission_df = pd.DataFrame({
    "filename": filenames,
    "category": predicted_labels
})

submission_df.to_csv("submission_ensemble_final.csv", index=False)
print("\n CSV de soumission créé : submission_ensemble_final.csv")